In [100]:
print("Hi and Welcome to the course!")

Hi and Welcome to the course!


In [101]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [102]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


# Let's make a quick call to a Frontier model to get started, as a preview!

In [103]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

In [104]:
from ollama import chat

from ollama import ChatResponse
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(model='gemma3:270m', messages=[
  {
    'role': 'user',
    'content': 'Why is the sky blue?',
  },
])
print(response['message']['content'])
# or access fields directly from the response object
print(response.message.content)

The sky is blue because of a phenomenon called **Rayleigh Scattering**. Here's the breakdown:

*   **Light Waves:** When light from the sun interacts with matter (like dust, water, or gases), it collides with these particles.
*   **Scattering:** The scattered light has different wavelengths, depending on the material it's coming from. The shorter wavelengths (blue and violet) are scattered more than longer wavelengths (red and orange).
*   **Reflection:** Some of the scattered light then reflects off the surface of the object.
*   **Entering the Atmosphere:**  The light from the sun is then absorbed by the Earth's atmosphere.
*   **Absorption:**  The atmosphere scatters the light.  The amount of scattering depends on the composition of the atmosphere and the distance light travels through.
*   **Blue Light is Scattered:**  The blue light is scattered more than the other colors.  This is why we see blue skies.

So, in short, the sky is blue because of the scattering of light by the Eart

## OK onwards with our first project

In [105]:
# Let's try out this utility

ed = fetch_website_contents("https://www.google.com")
print(ed)

Google

About
Store
Gmail
Images
Sign in
Upload image
Upload file
AI Mode
🍌
Create Image
AI Mode
AI Mode
🍌
Create Image
See more
Delete
Delete
Report inappropriate predictions
Cannot upload. Use a file in one of these formats: .avif, .bmp, .jpeg, .pdf, .png, .webp”
Google offered in:
हिन्दी
বাংলা
తెలుగు
मराठी
தமிழ்
ગુજરાતી
ಕನ್ನಡ
മലയാളം
ਪੰਜਾਬੀ
India
Advertising
Business
How Search works
Privacy
Terms
Settings
Search settings
Advanced search
Your data in Search
Search history
Search help
Send feedback
Dark theme: Off
Google apps


## Types of prompts

You may know this already - but if not, you will get very familiar with it!

Models like GPT have been trained to receive instructions in a particular way.

They expect to receive:

**A system prompt** that tells them what task they are performing and what tone they should use

**A user prompt** -- the conversation starter that they should reply to

In [106]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website
and provides a short, humorous summary.

Rules:
- Use markdown bullet points
- Start each line with '- '
- Write 10 to 15 bullet points
- Do not explain the rules
- print each point on a new line
"""

In [107]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [108]:
messages = [
    {"role": "system", "content": "alway reply in french"},
    {"role": "user", "content": "hi, how are you?"}
]
response : ChatResponse = chat(model='gemma3:270m', messages=messages)
print(response.message.content)

Bonjour ! Comment allez-vous ?



## And now let's build useful messages for gemma3:270m, using a function

In [109]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [110]:
# Try this out, and then try for a few more websites

# messages_for(ed)

## Time to bring it together - the API for OpenAI is very simple!

In [111]:
# And now: call the Ollama API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response: ChatResponse = chat(
        model = "gemma3:270m",
        messages = messages_for(website)
    )
    return response.message.content

    # return response.choices[0].message.content

In [112]:
response = summarize("https://vaaree.com/")
print(response)


Okay, I'm ready to analyze the website content and provide a short summary based on the provided information. I will strive to be informative and humorous while adhering to the rules. Let's begin!


In [113]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [114]:
display_summary("https://vaaree.com/")

Here's a short, humorous summary of the website:

"Buy Best Home Decor Items & Essentials Online At Best Prices - Vaaree"


# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [115]:
display_summary("https://cnn.com")

```
-
```


In [116]:
display_summary("https://anthropic.com")

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float32
)

pipe.enable_attention_slicing()

device = "mps" if torch.backends.mps.is_available() else "cpu"
pipe = pipe.to(device)

prompt = "A cute cat, soft lighting"

image = pipe(
    prompt,
    num_inference_steps=15,
    height=384,
    width=384
).images[0]

image.save("cat.png")
print("Image saved as cat.png")
